#  Customer Churn Prediction System
**Project Overview:** This project demonstrates an end-to-end Machine Learning pipeline to predict customer churn. It covers realistic data generation, exploratory data analysis (EDA), advanced model training with hyperparameter tuning, and a live web application deployment.

# Initial Setup & Library Installation

In [ ]:
# Install the Faker library to help us create our dataset
!pip install Faker

# Import all the necessary tools for our dataset creation
import pandas as pd
import numpy as np
from faker import Faker
import random

# Initialize Faker to start generating data
fake = Faker()

## Synthetic Dataset Generation

In [ ]:
# Create an empty list to store our data
data = []
num_records = 1500

# Lists for random selections
cities = ['Rawalpindi', 'Islamabad', 'Lahore', 'Karachi', 'Peshawar', 'Multan', 'Faisalabad', 'Quetta']
subscription_types = ['Basic', 'Standard', 'Premium']

print("Generating 1500 customer records with clear patterns....")

for i in range(num_records):
    # Basic Demographics
    customer_id = f"CUST-{10000 + i}"
    age = random.randint(18, 70)
    gender = random.choice(['Male', 'Female'])
    city = random.choice(cities)

    # Account Information
    subscription_type = random.choice(subscription_types)
    if subscription_type == 'Basic':
        monthly_spending = round(random.uniform(10.0, 30.0), 2)
    elif subscription_type == 'Standard':
        monthly_spending = round(random.uniform(31.0, 70.0), 2)
    else:
        monthly_spending = round(random.uniform(71.0, 120.0), 2)

    tenure = random.randint(1, 72)
    num_purchases = random.randint(1, 50)
    support_requests = random.randint(0, 10)
    login_frequency = random.randint(1, 30)
    last_activity_date = fake.date_between(start_date='-1y', end_date='today')
    satisfaction_score = random.randint(1, 5)

    #  Logic for Churn Status
    if satisfaction_score <= 2 and support_requests >= 4:
        churn_status = 1  # Unhappy customers with complaints definitely leave
    elif tenure < 6 and monthly_spending > 80:
        churn_status = 1  # New customers with high bills tend to leave early
    elif satisfaction_score >= 4 and tenure > 12:
        churn_status = 0  # Happy, long-term customers stay
    elif support_requests == 0 and login_frequency > 15:
        churn_status = 0  # Active customers with no issues stay
    else:
        # Slight randomness
        churn_probability = 0.15
        if satisfaction_score <= 3:
            churn_probability += 0.3
        if support_requests > 3:
            churn_probability += 0.3
        churn_status = 1 if random.random() < churn_probability else 0

    # Add row to data
    data.append([
        customer_id, age, gender, city, subscription_type,
        monthly_spending, tenure, num_purchases, support_requests,
        login_frequency, last_activity_date, satisfaction_score, churn_status
    ])

# Convert to Pandas DataFrame
columns = [
    'Customer ID', 'Age', 'Gender', 'City', 'Subscription Type',
    'Monthly Spending', 'Tenure', 'Number of Purchases', 'Customer Support Requests',
    'Login Frequency', 'Last Activity Date', 'Satisfaction Score', 'Churn Status'
]
df = pd.DataFrame(data, columns=columns)

print("Highly optimized dataset created successfully! Preview:")
display(df.head())

## Saving Dataset into CSV File

In [ ]:
# Save the dataframe to a CSV file
csv_filename = 'Custom_Customer_Churn_Dataset.csv'
df.to_csv(csv_filename, index=False)

print(f"Dataset saved successfully as '{csv_filename}'!")

## Exploratory Data Analysis (EDA) & Preprocessing

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# Create a figure with multiple subplots for EDA
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

#Churn Distribution (Pie Chart)
churn_counts = df['Churn Status'].value_counts()
axes[0, 0].pie(churn_counts, labels=['Retained (0)', 'Churned (1)'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0, 0].set_title('Overall Churn Distribution', fontsize=14, fontweight='bold')

# 2. Churn by Subscription Type
sns.countplot(data=df, x='Subscription Type', hue='Churn Status', palette=['#2ecc71', '#e74c3c'], ax=axes[0, 1])
axes[0, 1].set_title('Churn by Subscription Type', fontsize=14, fontweight='bold')

# Churn by Satisfaction Score
sns.countplot(data=df, x='Satisfaction Score', hue='Churn Status', palette=['#2ecc71', '#e74c3c'], ax=axes[1, 0])
axes[1, 0].set_title('Churn by Satisfaction Score', fontsize=14, fontweight='bold')

# Tenure vs Churn
sns.boxplot(data=df, x='Churn Status', y='Tenure', palette=['#2ecc71', '#e74c3c'], ax=axes[1, 1])
axes[1, 1].set_title('Customer Tenure vs Churn', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

#Correlation Heatmap for Numerical features
plt.figure(figsize=(10, 6))
numerical_cols = ['Age', 'Monthly Spending', 'Tenure', 'Number of Purchases', 'Customer Support Requests', 'Login Frequency', 'Satisfaction Score', 'Churn Status']
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.show()

## Data Preprocessing & Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Starting Data Preparation...")

#Drop columns that are not directly useful for the ML model's math
df_ml = df.drop(['Customer ID', 'Last Activity Date'], axis=1)

# Encode Categorical Variables
# Convert Gender to 0 and 1 using Label Encoder
le = LabelEncoder()
df_ml['Gender'] = le.fit_transform(df_ml['Gender'])

# Convert City and Subscription Type into multiple binary columns (One-Hot Encoding)
df_ml = pd.get_dummies(df_ml, columns=['City', 'Subscription Type'], drop_first=True)

# Separate Features (X) and Target Variable (y)
X = df_ml.drop('Churn Status', axis=1)
y = df_ml['Churn Status']

# Train-Test Split (80% data for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nData Preparation & Encoding Successful!")
print(f"Training Data Shape (Rows, Columns): {X_train.shape}")
print(f"Testing Data Shape (Rows, Columns): {X_test.shape}")
display(X_train.head())

##  Model Training & Evaluation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

print("Optimizing and Training Models to boost accuracy.....")

# Hyperparameter Tuning for Random Forest
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced'] # Helps balance the churned vs retained data
}

print("Tuning Random Forest for Maximum Accuracy...")
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=3, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
best_rf_model = rf_grid.best_estimator_

# Hyperparameter Tuning for XGBoost
xgb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7]
}

print("Tuning XGBoost for Maximum Accuracy...")
xgb_grid = GridSearchCV(xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42), xgb_params, cv=3, scoring='accuracy', n_jobs=-1)
xgb_grid.fit(X_train, y_train)
best_xgb_model = xgb_grid.best_estimator_

# Make Predictions
rf_pred = best_rf_model.predict(X_test)
rf_prob = best_rf_model.predict_proba(X_test)[:, 1]

xgb_pred = best_xgb_model.predict(X_test)
xgb_prob = best_xgb_model.predict_proba(X_test)[:, 1]

# Calculate Metrics
def get_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

rf_metrics = get_metrics(y_test, rf_pred, rf_prob)
xgb_metrics = get_metrics(y_test, xgb_pred, xgb_prob)

# Create Comparison Table
metrics_df = pd.DataFrame([rf_metrics, xgb_metrics], index=['Random Forest', 'XGBoost'])
print("\n--- Model Comparison Results ---")
display(metrics_df)

# Visualize Comparison
sns.set_theme(style="whitegrid")
metrics_df.T.plot(kind='bar', figsize=(10, 6), colormap='magma', edgecolor='black')
plt.title('Random Forest vs XGBoost Performance', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)
plt.xticks(rotation=0, fontsize=11)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Save the Models
joblib.dump(best_rf_model, 'rf_model.pkl')
joblib.dump(best_xgb_model, 'xgb_model.pkl')
joblib.dump(list(X_train.columns), 'model_features.pkl')

print("\nModels  saved successfully!!!!")

## Live Prediction Interface (Gradio App)

In [ ]:
# Install Gradio
!pip install gradio -q

import gradio as gr
import pandas as pd
import joblib

# Load the trained model and features
model = joblib.load('rf_model.pkl')
model_features = joblib.load('model_features.pkl')

# Define the Prediction Logic
def predict_churn(age, gender, city, sub_type, monthly_spending, tenure, num_purchases, support_requests, login_freq, satisfaction):
    # Create a dictionary perfectly matched to model features with 0s
    input_dict = {feature: 0 for feature in model_features}

    # Fill in numerical data
    input_dict['Age'] = age
    input_dict['Tenure'] = tenure
    input_dict['Monthly Spending'] = monthly_spending
    input_dict['Number of Purchases'] = num_purchases
    input_dict['Customer Support Requests'] = support_requests
    input_dict['Login Frequency'] = login_freq
    input_dict['Satisfaction Score'] = satisfaction

    # Fill Categorical Data safely
    input_dict['Gender'] = 1 if gender == "Male" else 0

    if f'City_{city}' in input_dict:
        input_dict[f'City_{city}'] = 1

    if f'Subscription Type_{sub_type}' in input_dict:
        input_dict[f'Subscription Type_{sub_type}'] = 1

    # Convert exactly to DataFrame
    input_df = pd.DataFrame([input_dict])

    # Predict
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]

    if prediction == 1:
        return f"⚠️ HIGH RISK (Churn)\nProbability: {probability:.1%}\nInsight: Action required to retain this customer."
    else:
        return f"✅ SAFE (Retained)\nProbability: {probability:.1%}\nInsight: Customer profile is stable."

# Build the UI
interface = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Number(label="Age", value=30),
        gr.Radio(choices=["Male", "Female"], label="Gender", value="Male"),
        gr.Dropdown(choices=['Rawalpindi', 'Islamabad', 'Lahore', 'Karachi', 'Peshawar', 'Multan', 'Faisalabad', 'Quetta'], label="City", value="Islamabad"),
        gr.Dropdown(choices=["Basic", "Standard", "Premium"], label="Subscription Type", value="Standard"),
        gr.Number(label="Monthly Spending ($)", value=50.0),
        gr.Number(label="Tenure (Months)", value=12),
        gr.Number(label="Number of Purchases", value=5),
        gr.Number(label="Support Requests", value=2),
        gr.Number(label="Login Frequency (per month)", value=10),
        gr.Slider(minimum=1, maximum=5, step=1, label="Satisfaction Score", value=3)
    ],
    outputs=gr.Textbox(label="Prediction Result", lines=3),
    title="🚀 Customer Churn Prediction System",
    description="Enter customer details to instantly predict if they are at risk of leaving Teyzix Core services."
)

# Launch inside Colab with a public shareable link
interface.launch(share=True)